# 03 — Exploratory Data Analysis

> **Food Delivery Analytics Project**  
> Detailed EDA: univariate, bivariate, and multivariate analysis across orders,
> revenue, restaurants, ratings, and delivery performance.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src import analysis as an
from src.utils import load_processed, fmt_inr, MONTH_ORDER, DAY_ORDER

pd.set_option('display.max_columns', 50)
plt.rcParams.update({'figure.figsize': (11, 5), 'figure.dpi': 100})
sns.set_theme(style='whitegrid', palette='Blues_r')

df = load_processed('../data/processed/cleaned_food_delivery_orders.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 1. Univariate Analysis
### 1a. Order Status Distribution

In [ ]:
status = an.order_status_distribution(df)
colors = {'Delivered': '#16a34a', 'Cancelled': '#dc2626', 'Pending': '#f97316'}
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(status['count'], labels=status['order_status'],
       colors=[colors[s] for s in status['order_status']],
       autopct='%1.1f%%', startangle=90, pctdistance=0.85)
ax.set_title('Order Status Distribution')
plt.tight_layout()
plt.show()
print(status.to_string(index=False))

### 1b. Order Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
delivered = df[df['order_status'] == 'Delivered']
axes[0].hist(delivered['order_amount'], bins=40, color='#1a3a5c', edgecolor='white')
axes[0].set_title('Order Amount Distribution')
axes[0].set_xlabel('Order Amount (₹)')
axes[0].set_ylabel('Count')

axes[1].hist(delivered['final_amount'], bins=40, color='#0d9488', edgecolor='white')
axes[1].set_title('Final Amount (after Discount) Distribution')
axes[1].set_xlabel('Final Amount (₹)')
plt.suptitle('Order Value Distributions — Delivered Orders')
plt.tight_layout()
plt.show()

print('Order Amount:')
print(delivered['order_amount'].describe().round(2).to_string())
print('\nFinal Amount:')
print(delivered['final_amount'].describe().round(2).to_string())

### 1c. Rating Distribution

In [ ]:
rating_dist = an.rating_distribution(df)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(rating_dist['customer_rating'].astype(str), rating_dist['count'], color='#f97316')
ax.set_title('Customer Rating Distribution')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
plt.show()
print(f'Mean rating : {df["customer_rating"].mean():.2f}')
print(f'Median rating: {df["customer_rating"].median()}')

## 2. Bivariate Analysis
### 2a. Orders by Month

In [ ]:
monthly = an.orders_by_month(df)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.bar(monthly['month_name'], monthly['orders'], color='#1a3a5c', alpha=0.7, label='Orders')
ax2.plot(monthly['month_name'], monthly['revenue'] / 1e5, color='#f97316', marker='o', label='Revenue (₹L)')
ax1.set_title('Monthly Orders & Revenue')
ax1.set_xlabel('Month')
ax1.set_ylabel('Orders')
ax2.set_ylabel('Revenue (₹ Lakhs)')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 2b. Revenue by Cuisine

In [ ]:
rev_cuisine = an.revenue_by_cuisine(df)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(rev_cuisine['cuisine'], rev_cuisine['revenue'] / 1e5, color='#0d9488')
ax.set_xlabel('Revenue (₹ Lakhs)')
ax.set_title('Revenue by Cuisine Category')
plt.tight_layout()
plt.show()

### 2c. Delivery Time Distribution

In [ ]:
delivered = df[df['order_status'] == 'Delivered']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(delivered['delivery_time_minutes'].dropna(), bins=30, color='#1a3a5c', edgecolor='white')
axes[0].axvline(delivered['delivery_time_minutes'].median(), color='#f97316', linestyle='--', label=f'Median: {delivered["delivery_time_minutes"].median():.0f} min')
axes[0].set_title('Actual Delivery Time Distribution')
axes[0].set_xlabel('Minutes')
axes[0].legend()

axes[1].hist(delivered['delivery_delay'].dropna(), bins=30, color='#dc2626', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='#16a34a', linestyle='--', label='On Time')
axes[1].set_title('Delivery Delay Distribution (Actual − Estimated)')
axes[1].set_xlabel('Delay (minutes)')
axes[1].legend()
plt.tight_layout()
plt.show()
print(f'Avg delay: {delivered["delivery_delay"].mean():.1f} min')
print(f'On-time %: {(delivered["delivery_delay"] <= 0).mean()*100:.1f}%')

## 3. Multivariate Analysis
### 3a. Rating vs Delivery Time

In [ ]:
rtd = an.rating_vs_delivery_time(df, bins=8)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rtd['dt_bin_mid'], rtd['avg_rating'], marker='o', color='#1a3a5c', linewidth=2)
ax.fill_between(rtd['dt_bin_mid'], rtd['avg_rating'], alpha=0.15, color='#1a3a5c')
ax.set_xlabel('Delivery Time (minutes)')
ax.set_ylabel('Average Customer Rating')
ax.set_title('Longer Delivery Time → Lower Customer Rating')
ax.set_ylim(1, 5)
plt.tight_layout()
plt.show()

### 3b. Correlation Matrix

In [ ]:
corr = an.correlation_matrix(df)
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix — Key Business Metrics')
plt.tight_layout()
plt.show()

### 3c. Revenue vs Rating by Cuisine (Bubble Chart)

In [ ]:
scatter_df = an.revenue_vs_rating_scatter(df)
fig = px.scatter(
    scatter_df, x='avg_rating', y='revenue',
    size='orders', color='cuisine',
    hover_name='restaurant_name',
    title='Revenue vs Rating per Restaurant (bubble = order volume)',
    size_max=40
)
fig.show()

## 4. Key Observations

1. **~88% of orders are Delivered** — cancellation rate is ~8%, within industry norms.
2. **Avg order value ≈ ₹380-420** for delivered orders; discounts average ~15% of order amount.
3. **Lunch (11–14h) and Dinner (18–22h)** are the peak ordering windows.
4. **Weekends are 25-30% busier** than weekdays.
5. **North Indian and Chinese cuisines** dominate both order volume and revenue.
6. **Delivery times** average ~38 minutes; most orders land in 25–50 minutes.
7. **Strong negative correlation** between `delivery_delay` and `customer_rating`.
8. **Mumbai and Delhi** are the top-revenue cities, consistent with population size.

**Next step:** `04_business_analysis.ipynb`